# dotLLM — dual-CUDA pipeline-parallel (layer-spanning) validation (#367)

Validates `CudaPipelineTransformerModel` (#367): a transformer split across **two CUDA devices** —
stage-0 = layers `[0..K)` on GPU 0, stage-1 = layers `[K..L)` + final norm + LM head on GPU 1 — with
the hidden state handed off device0 -> host (FP32) -> device1. This is the CUDA mirror of the shipped
Vulkan dual-device pipeline (#366).

**Before running:** Settings -> Accelerator -> **GPU T4 x2**, and Settings -> Internet -> **On**.

Run the cells top-to-bottom. The single-device split theories prove the windowed-weight + resume-from-
hidden machinery on one GPU; the `CrossDevice*` theories prove the real two-GPU span (auto-skip on 1 GPU).
Logits must match the un-split `CudaTransformerModel` to a tight band (abs 5e-3 / rel 5e-3).

In [ ]:
%%bash
# Bootstrap: clone the #367 branch so kaggle/setup.sh is available.
export DOTLLM_REPO="${DOTLLM_REPO:-https://github.com/jamesburton/dotLLM.git}"
export DOTLLM_BRANCH="${DOTLLM_BRANCH:-issue/367-cuda-pipeline-parallel}"
cd /kaggle/working
rm -rf dotLLM
git clone --depth 1 --branch "$DOTLLM_BRANCH" "$DOTLLM_REPO"
echo "cloned $DOTLLM_BRANCH from $DOTLLM_REPO"

In [ ]:
%%bash
# Sanity: OS, dual T4s, nvcc.
export DOTLLM_BRANCH="${DOTLLM_BRANCH:-issue/367-cuda-pipeline-parallel}"
bash /kaggle/working/dotLLM/kaggle/setup.sh env

In [ ]:
%%bash
# Install .NET 10 SDK into $HOME (no root).
export DOTLLM_BRANCH="${DOTLLM_BRANCH:-issue/367-cuda-pipeline-parallel}"
bash /kaggle/working/dotLLM/kaggle/setup.sh dotnet

In [ ]:
%%bash
# Compile CUDA kernels -> PTX (compute_75; T4 = sm_75) with the LOCAL toolkit (avoids CUDA error 222).
export DOTLLM_BRANCH="${DOTLLM_BRANCH:-issue/367-cuda-pipeline-parallel}"
bash /kaggle/working/dotLLM/kaggle/setup.sh ptx

In [ ]:
%%bash
# Restore + build the solution (Release); re-syncs driver-matched PTX into bin/.
export DOTLLM_BRANCH="${DOTLLM_BRANCH:-issue/367-cuda-pipeline-parallel}"
bash /kaggle/working/dotLLM/kaggle/setup.sh build

## Pipeline-parallel parity (the #367 goal)

`CudaPipelineParityTests` runs `splits {1,2,3}` over a 4-layer synthetic dense fixture:

- **`PipelineModel_*` (single-device):** both stages on GPU 0 (two contexts) — isolates windowed
  weights + per-stage layer reindex + resume-from-hidden without needing two GPUs.
- **`CrossDevicePipelineModel_*` (dual-device):** stage-0 on GPU 0, stage-1 on GPU 1 — the real span
  (device0 -> host -> device1). Skips automatically when `CudaDevice.GetDeviceCount() < 2`.

Prefill and decode (prefill + one KV-cached decode step) are both covered.

In [ ]:
%%bash
export DOTLLM_BRANCH="${DOTLLM_BRANCH:-issue/367-cuda-pipeline-parallel}"
bash /kaggle/working/dotLLM/kaggle/setup.sh test-pipeline

## Notes

- On a **single**-GPU session the `CrossDevice*` theories skip and only the single-device split runs;
  enable **GPU T4 x2** to exercise the genuine device0 -> host -> device1 handoff.
- The composite KV-cache (`CudaPipelineKvCache`) keeps each stage's FP16 KV table on its own device and
  routes global layer -> (stage, local index). Cross-device parity is to FP16 precision (same as a
  single-device CUDA run); the FP32 hidden-state round-trip at the boundary is lossless.
- M-scope: dense / GQA causal (no MLA / MoE / gemma4 / graph-capture), matching the Vulkan #366 scope.
- Override repo/branch with `DOTLLM_REPO` / `DOTLLM_BRANCH`.